# Week 4 — Fine-tune TrOCR + evaluate with CER/WER

**Before running any code this week: Runtime > Change runtime type > GPU.**

This notebook drives `src/train.py` and `src/evaluate.py`. See `src/train.py`'s
docstring for the full list of changes vs. the handout's Cells 1-3 (Seq2SeqTrainer
with `predict_with_generate`, cosine LR + warmup, more epochs with early
stopping on CER, mixed precision, beam-search decoding). The short version:
these changes exist because they measurably reduce Character Error Rate, not
just to make the code longer.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch
# -- see Quickstart_Colab.ipynb's Cell 1 for why (an unqualified clone/pull
# can silently track a branch that's missing later fixes, and `git pull`
# then reports "Already up to date" against the wrong branch with no
# warning). Replace with your own fork's URL/branch if running elsewhere,
# or with 'main' once this branch has actually been merged there.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. The actual training/eval scripts
# run as a separate `!python ...` subprocess each time, so they always
# see what's genuinely on disk -- this mirrors that rather than this
# kernel's own (possibly stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )

# Functional smoke test, not just a version check -- and run in a genuinely
# fresh subprocess, not this notebook kernel. Pillow (and any package with a
# compiled C extension) cannot have its native .so swapped out in an
# already-running process even after a correct on-disk reinstall; the only
# way to know it will actually work is to check from a brand new process,
# which is also exactly what the training/eval scripts do.
pil_check = subprocess.run(
    ['python3', '-c',
     "from PIL import Image; im = Image.new('RGB', (32, 32), 'white'); "
     "im.save('/tmp/_pil_smoke_test.png'); "
     "Image.open('/tmp/_pil_smoke_test.png').load(); print('PIL round-trip OK')"],
    capture_output=True, text=True,
)
print(pil_check.stdout.strip())
if pil_check.returncode != 0:
    raise RuntimeError(
        'Pillow is installed but broken (fails to save/load an image even in '
        'a fresh process) -- this is not a version-pin issue, it is a '
        'corrupted install, most likely leftover mixed files on disk from '
        'earlier pip installs/reinstalls in this same Colab VM. '
        '`Runtime > Restart session` will NOT fix this (it only restarts '
        'the Python process, not the disk). You need '
        '`Runtime > Disconnect and delete runtime`, then reconnect to a '
        'brand new VM and run this notebook from Cell 1 on that clean '
        'slate.\n\nOriginal error:\n' + pil_check.stderr[-1500:]
    )

print('Setup complete -- all pinned versions correctly installed on disk and working.')


In [ ]:
import torch
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')


## Train

Adjust `--epochs` / `--batch-size` for your Colab tier. Training logs CER/WER after every epoch (unlike the handout's loop, which only prints training loss).

In [ ]:
!python src/train.py \
    --train-csv data/train.csv --test-csv data/test.csv \
    --output-dir /content/drive/MyDrive/SI26-urdu-ocr-model \
    --epochs 12 --batch-size 8 --grad-accum 2


(If you'd rather keep checkpoints local to the Colab VM instead of Drive, drop the `/content/drive/...` path and use `models/urdu-trocr` — just remember to save to Drive/HF Hub before the session ends.)

## Evaluate

Reports Character Error Rate, Word Error Rate, and exact-match accuracy (the handout's original metric) side by side, plus the 5 worst predictions for your write-up.

In [ ]:
!python src/evaluate.py /content/drive/MyDrive/SI26-urdu-ocr-model/final \
    --test-csv data/test.csv \
    --report-csv reports/eval_report.csv


**What to record** (same ask as the handout, now with real numbers instead of just exact-match): your CER / WER / exact-match accuracy, and 3-5 wrong examples from `reports/eval_report.csv` (sort by the `cer` column). These go into your Week 5 README's Results section.

## Save your model

Already saved to Drive by `--output-dir` above during training. To also publish it to the HuggingFace Hub (recommended — Spaces deployment in Week 5 can then load it with `MODEL_PATH=yourusername/your-model-name`, no Drive mount needed):

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a HuggingFace token with write access

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_dir = '/content/drive/MyDrive/SI26-urdu-ocr-model/final'
repo_id = 'yourusername/urdu-trocr-si26'  # <-- change this

model = VisionEncoderDecoderModel.from_pretrained(model_dir)
processor = TrOCRProcessor.from_pretrained(model_dir)
model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)
print(f'Model published at: https://huggingface.co/{repo_id}')
